In [2]:
#import required libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, trim
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import StringIndexer,VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [3]:
#2. intitialize spark session
spark=SparkSession.builder.appName("Ecommerce purchase history").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/18 02:39:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
#3. load dataset
data = spark.read.csv("/kaggle/input/datasets/mkechinov/ecommerce-purchase-history-from-electronics-store/kz.csv",header=True,inferSchema=True)
print("Dataset schema:")
data.printSchema()

Dataset schema:
root
 |-- event_time: timestamp (nullable = true)
 |-- order_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: long (nullable = true)



In [18]:
#4. Data preprocessing
data=data.withColumn("price",when(col("price")==" ",None).otherwise(trim(col("price"))))
data=data.withColumn("price",col("price").cast(DoubleType()))
data=data.fillna({"price":0.0})

In [17]:
#5 convert categorical columns to numeric

#drop existing indexed columns if they exist to allow re-running the cell
coloumns_to_drop=["category_id","category_code","brand"]

for col_name in coloumns_to_drop:
  if col_name in data.columns:
    data=data.drop(col_name)

#create indexers
category_id=StringIndexer(inputCol="ID",outputCol="category_id",handleInvalid="keep")
category_code=StringIndexer(inputCol="Code",outputCol="category_code",handleInvalid="keep")
brand=StringIndexer(inputCol="Brand",outputCol="brand",handleInvalid="keep")

#apply transformation
data=category_id.fit(data).transform(data)
data=category_code.fit(data).transform(data)
data=brand.fit(data).transform(data)

#show result
data.select("ID","category_id","Code","category_code","Brand","brand").show(9)

Py4JJavaError: An error occurred while calling o226.fit.
: org.apache.spark.SparkException: Input column ID does not exist.
	at org.apache.spark.ml.feature.StringIndexerBase.$anonfun$validateAndTransformSchema$2(StringIndexer.scala:134)
	at scala.collection.ArrayOps$.flatMap$extension(ArrayOps.scala:971)
	at org.apache.spark.ml.feature.StringIndexerBase.validateAndTransformSchema(StringIndexer.scala:122)
	at org.apache.spark.ml.feature.StringIndexerBase.validateAndTransformSchema$(StringIndexer.scala:114)
	at org.apache.spark.ml.feature.StringIndexer.validateAndTransformSchema(StringIndexer.scala:152)
	at org.apache.spark.ml.feature.StringIndexer.transformSchema(StringIndexer.scala:251)
	at org.apache.spark.ml.PipelineStage.transformSchema(Pipeline.scala:72)
	at org.apache.spark.ml.feature.StringIndexer.fit(StringIndexer.scala:236)
	at org.apache.spark.ml.feature.StringIndexer.fit(StringIndexer.scala:152)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method

In [ ]:
#6. Feature Selection
feature_columns=["tenure","MonthlyCharges","TotalCharges","gender_index","contract_index"]
assembler = VectorAssembler(inputCols=feature_columns,outputCol="features")

#Drop the 'features'column if it already exits to prevent error on re-run
if "features" in data.columns:
  data=data.drop('features')
data=assembler.transform(data)

In [ ]:
#7 split dataset
#Training (70%), Validation (15%), Testing (15%)
train_data, val_data, test_data = data.randomSplit([0.7,0.15,0.15],seed=42)
print("Training Data:",train_data.count())
print("Validation Data:",val_data.count())
print("Testing Data:",test_data.count())

In [ ]:
#8 Train logistic regression model
lr = LogisticRegression(featuresCol="features",labelCol="churn_index",maxIter=10)
model = lr.fit(train_data)
print("Model Training Completed")

In [ ]:
#9 validation performance
val_predictions = model.transform(val_data)
evaluator = MulticlassClassificationEvaluator(labelCol="churn_index",predictionCol="prediction")
accuracy_val = evaluator.setMetricName("accuracy").evaluate(val_predictions)
precision_val = evaluator.setMetricName("weightedPrecision").evaluate(val_predictions)
recall_val = evaluator.setMetricName("weightedRecall").evaluate(val_predictions)

print("\nValidation Results")
print("-------------------")
print("Accuracy:",accuracy_val)
print("Precision:",precision_val)
print("Recall:",recall_val)

In [ ]:
#10 test model performance
test_predictions = model.trasnform(test_data)
accuracy_test = evaluator.setMetricName("accuracy").evaluate(test_predictions)
precision_test = evaluator.setMetricName("weightedPrecision").evaluate(test_predictions)
recall_test = evaluator.setMetricName("weightedRecall").evaluate(test_predictions)

print("\nTest Results")
print("-------------")
print("Accuracy:",accuracy_test)
print("Precision:",precision_test)
print("Recall:",recall_test)

In [ ]:
#11 show sample predictions
test_predictions.select("customerID","Churn","prediction").show(9)

In [ ]:
#12 stop spark session
spark.stop()